In [3]:
from functools import partial

from ultralytics import settings

from edge_detector.backbones.custom import (
    C2fBackbone,
    FasterNetBackbone,
    RepVGGBackbone,
    ShuffleNetBackbone,
)
from edge_detector.backbones.baselines import create_baseline_backbone
from edge_detector.scripts.trainer import CustomDetectionTrainer

In [4]:
settings.update({
    "clearml": False,
    "tensorboard": False,
})

In [5]:
BACKBONES = {
    "custom_shufflenet": ShuffleNetBackbone,
    "custom_fasternet": FasterNetBackbone,
    "custom_repvgg": RepVGGBackbone,
    "custom_c2f": C2fBackbone,

    "baseline_shufflenetv2_x0_5": partial(
        create_baseline_backbone,
        model_name="shufflenet_v2_x0_5",
        out_channels=(48, 96, 192),
        deploy=False,
    ),
    "baseline_fasternet_t0": partial(
        create_baseline_backbone,
        model_name="fasternet_t0",
        out_channels=(80, 160, 320),
        deploy=False,
    ),
    "baseline_yolov8n": partial(
        create_baseline_backbone,
        model_name="yolov8n",
        out_channels=(64, 128, 256),
        deploy=False,
    ),
    "baseline_yolov6n": partial(
        create_baseline_backbone,
        model_name="yolov6n_efficientrep",
        out_channels=(64, 128, 256),
        deploy=False,
    ),
}

In [6]:
DATASET_YAML = "/home/kamynin.a3/diploma/dataset/dataset.yaml"
PROJECT_DIR = "/home/kamynin.a3/diploma/runs/models"

EPOCHS = 1

COMMON_OVERRIDES = {
    "model": "yolo11n.yaml",
    "data": DATASET_YAML,
    "epochs": EPOCHS,
    "batch": 32,
    "imgsz": 640,
    "close_mosaic": 10,
    "device": 0,
    "optimizer": "AdamW",
    "lr0": 0.01,
    "lrf": 0.01,
    "mosaic": 1.0,
    "project": PROJECT_DIR,
}

MODEL_OVERRIDES = {
    "baseline_fasternet_t0": {
        "amp": False,
    },
}

In [ ]:
def train_model(model_name: str):
    overrides = {
        **COMMON_OVERRIDES,
        **MODEL_OVERRIDES.get(model_name, {}),
        "name": f"{model_name}_{EPOCHS}ep",
    }

    trainer = CustomDetectionTrainer(
        backbone_cls=BACKBONES[model_name],
        overrides=overrides,
    )

    trainer.train()

    return trainer

In [ ]:
MODELS_TO_TRAIN = list(BACKBONES)

for index, model_name in enumerate(MODELS_TO_TRAIN, start=1):
    print(f"\n[{index}/{len(MODELS_TO_TRAIN)}] {model_name}")

    train_model(model_name)